# Pamoka 10 - DI agentai produkcijoje

Šioje pamokoje sužinosite apie **produkcinio panaudojimo modelius** DI agentams, naudojant Microsoft Agent Framework (Python). Apžvelgsime:

- **Stebimąjį gebėjimą** — laikų matavimą ir žurnalų fiksavimą agento sąveikoms
- **Vertinimą** — vertintojo agento naudojimą atsakymų kokybės įvertinimui
- **Sąnaudų valdymą** — strategijas žetonų optimizavimui ir modelio parinkimui

Scenarijus – **kelionių agentas**, kuris padeda naudotojams planuoti keliones, kartu integruojant stebėjimą ir vertinimą.


## Nustatymai


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import time
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Azure AI Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Gamybos svarstymai

Perkėlimas AI agentų nuo prototipų iki gamybos reikalauja atidaus dėmesio trims pagrindams:

1. **Stebėjimas** — Reikia matyti, ką agentas daro, kiek laiko tai trunka ir kokius įrankius jis naudoja. Be sekimo ir registravimo, gamybos problemų trikčių šalinimas yra beveik neįmanomas.

2. **Vertinimas** — Automatiniai kokybės patikrinimai užtikrina, kad agente atsakymai išlieka tikslūs, pilni ir naudingi laikui bėgant. Vertinimo agentas gali įvertinti atsakymus pagal apibrėžtus kriterijus.

3. **Išlaidų valdymas** — Žetonų naudojimas tiesiogiai veikia išlaidas. Tokios strategijos kaip užklausų optimizavimas, modelių pasirinkimas ir kešavimas padeda palaikyti išlaidas kontroliuojamas neprarandant kokybės.


## Stebimo agento kūrimas

Apibrėžiame kelionės įrankius ir apgaubiam agento kvietimą laiko matavimo mechanizmu, kad galėtume stebėti delsą. Gamyboje jūs integruotumėte su OpenTelemetry arba panašia sekimo sistema.


In [ ]:
@tool(approval_mode="never_require")
def get_flight_info(destination: Annotated[str, "The destination city"]) -> str:
    """Get flight information for a destination."""
    flights = {
        "Paris": "BA 304, 08:30-11:45, $350",
        "Tokyo": "JL 044, 11:00-07:00+1, $890",
        "Barcelona": "VY 7821, 07:15-10:30, $280",
    }
    return flights.get(destination, f"No flights found to {destination}")


@tool(approval_mode="never_require")
def get_activity_suggestions(destination: Annotated[str, "The destination city"]) -> str:
    """Get activity suggestions for a destination."""
    activities = {
        "Paris": "Louvre Museum, Eiffel Tower, Seine River Cruise, Montmartre walking tour",
        "Tokyo": "Senso-ji Temple, Tsukiji Market tour, Shibuya Crossing, teamLab Borderless",
        "Barcelona": "Sagrada Familia, Park Güell, La Boqueria Market, Gothic Quarter walk",
    }
    return activities.get(destination, f"No activities found for {destination}")

In [ ]:
agent = client.as_agent(
    tools=[get_flight_info, get_activity_suggestions],
    name="TravelAgent",
    instructions="You are a helpful travel agent. Use the available tools to help users plan their trips. Provide comprehensive, actionable travel advice.",
)

# Simple observability: track timing
start_time = time.time()
response = await agent.run(
    "I want to plan a day trip in Paris. What flights and activities do you recommend?",
    )
elapsed = time.time() - start_time
print(f"Response ({elapsed:.2f}s):\n{response}")

## Įvertinimo modeliai

Dažnas gamybos modelis yra naudoti antrą agentą kaip **vertintoją**. Vertintojas įvertina pagrindinio agento atsakymą pagal iš anksto nustatytus kriterijus, tokius kaip išsamumas, tikslumas ir naudingumas.

Tai leidžia:
- Automatinį kokybės patikrinimą prieš atsakymams pasiekiant vartotojus
- Regresijos nustatymą, kai keičiasi užklausos ar modeliai
- Nuolatinį agento veiklos stebėjimą laikui bėgant


In [ ]:
evaluator = client.as_agent(
    name="ResponseEvaluator",
    instructions="""You evaluate travel agent responses on these criteria:
1. Completeness (1-5): Did it cover flights AND activities?
2. Accuracy (1-5): Is the information consistent?
3. Helpfulness (1-5): Would a traveler find this actionable?
4. Overall Score (1-5)
Provide scores and a brief explanation for each.""",
)

evaluation = await evaluator.run(f"Evaluate this travel agent response:\n\n{response}")
print(f"Evaluation:\n{evaluation}")

## Išlaidų valdymo strategijos

Išlaidų valdymas yra labai svarbus gamybos AI agentams. Štai pagrindinės strategijos:

| Strategija | Aprašymas |
|---|---|
| **Užklausos optimizavimas** | Laikykite sistemos instrukcijas glaustas. Pašalinkite perteklinį kontekstą, kad sumažintumėte įvesties žetonų skaičių. |
| **Modelio pasirinkimas** | Naudokite mažesnius, pigesnius modelius (pvz., GPT-4o-mini) paprastoms užduotims, tokioms kaip klasifikavimas ar išgavimas, o sudėtingoms užduotims rezervuokite didesnius modelius. |
| **Kešavimas** | Kešuokite įrankių rezultatus ir dažnas užklausas, kad išvengtumėte perteklinių API kvietimų. |
| **Žetonų biudžetai** | Nustatykite `max_tokens` ribas, kad išvengtumėte netikėtai ilgų atsakymų. |
| **Grupavimas** | Jei įmanoma, sugrupuokite kelias vartotojo užklausas į vieną API kvietimą. |

Praktikoje gerai veikia sluoksniuota metodika: paprastas užklausas nukreipkite į greitą, nebrangų modelį, o sudėtingas - į pajėgesnį.


## Santrauka

Šiame pamokoje jūs sužinojote, kaip:

1. **Pridėti stebėjimą** agentų sąveikoms su laiko matavimu ir registravimu, sudarant pagrindą sekimui ir stebėsenai.
2. **Automatiškai įvertinti agentų atsakymus** naudojant vertinimo agentą, kuris vertina užbaigtumą, tikslumą ir naudingumą.
3. **Valdyti sąnaudas** optimizuojant užklausas, pasirenkant modelius, naudojant talpyklas ir valdyti žetonų biudžetus.

Šie gamybiniai modeliai padeda užtikrinti, kad jūsų AI agentai būtų patikimi, matuojami ir ekonomiški masteliu.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Atsakomybės apribojimas**:
Šis dokumentas buvo išverstas naudojant dirbtinio intelekto vertimo paslaugą [Co-op Translator](https://github.com/Azure/co-op-translator). Nors siekiame tikslumo, prašome atkreipti dėmesį, kad automatiniai vertimai gali turėti klaidų ar netikslumų. Originalus dokumentas jo gimtąja kalba laikomas autoritetingu šaltiniu. Svarbiai informacijai rekomenduojama naudoti profesionalų žmogiškąjį vertimą. Mes neatsakome už jokius nesusipratimus ar neteisingą interpretaciją, kilusią naudojantis šiuo vertimu.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
